# Async/Await + Async For + Yield — Fully Explained

This notebook covers:
1. What is `async/await` and why it exists
2. Sync vs Async — the difference
3. `async for` with `yield` (async generators)
4. Why `async` appears in 3 places
5. The chain rule

---
# PART A: async / await
---

## 1. The Problem: Waiting Wastes Time

Most programs spend time **waiting** — waiting for an API response, waiting for a database query, waiting for a file to load.

During that wait, the CPU does **nothing**. It just sits there.

```
Sync code:
  Call API ──── wait 2s doing nothing ──── got response
  Call API ──── wait 2s doing nothing ──── got response
  Call API ──── wait 2s doing nothing ──── got response
  Total: 6 seconds

Async code:
  Call API 1 ─┐
  Call API 2 ─┼── all waiting together ──── all responses arrive
  Call API 3 ─┘
  Total: 2 seconds
```

## 2. The Analogy: Restaurant Kitchen

**Sync (blocking):**
- Chef puts bread in toaster
- Stares at toaster for 3 minutes doing NOTHING
- Toast done. Now starts boiling eggs.
- Stares at pot for 5 minutes doing NOTHING
- Eggs done. Total: 8 minutes.

**Async (non-blocking):**
- Chef puts bread in toaster
- While toaster runs, starts boiling eggs
- While both run, preps the juice
- Toast done → grab it. Eggs done → grab them.
- Total: 5 minutes (everything overlapped)

## 3. `await` = "Go do this, come back when it's done. Meanwhile I'm free."

In [11]:
import asyncio
import time

# ================================
# SYNC version: blocks during wait
# ================================
def make_breakfast_sync():
    print("  Starting toast...")
    time.sleep(2)  # BLOCKS. Can't do anything else.
    print("  Toast done!")
    
    print("  Starting eggs...")
    time.sleep(3)  # BLOCKS again.
    print("  Eggs done!")

print("SYNC breakfast:")
start = time.time()
make_breakfast_sync()
print(f"  Total: {time.time() - start:.1f}s (toast + eggs = sequential)\n")

SYNC breakfast:
  Starting toast...
  Toast done!
  Starting eggs...
  Eggs done!
  Total: 5.0s (toast + eggs = sequential)



In [12]:
# =================================
# ASYNC version: free during waits
# =================================
async def make_toast():
    print("  Starting toast...")
    await asyncio.sleep(2)  # "Go toast. I'll do other stuff. Call me when done."
    print("  Toast done!")

async def make_eggs():
    print("  Starting eggs...")
    await asyncio.sleep(3)  # "Go boil. I'll do other stuff."
    print("  Eggs done!")

async def make_breakfast_async():
    # Start BOTH at the same time
    await asyncio.gather(make_toast(), make_eggs())

print("ASYNC breakfast:")
start = time.time()
await make_breakfast_async()
print(f"  Total: {time.time() - start:.1f}s (toast AND eggs = parallel)")
print("\n  Same work, but overlapped the waits!")

ASYNC breakfast:
  Starting toast...
  Starting eggs...
  Toast done!
  Eggs done!
  Total: 3.0s (toast AND eggs = parallel)

  Same work, but overlapped the waits!


## 4. The Rules of async/await

**Rule 1:** To use `await`, the function MUST be `async def`
```python
async def my_function():    # Must be async
    result = await something()  # Then you can await
```

**Rule 2:** `await` can only be used on "awaitable" things (async functions, sleep, network calls)
```python
await asyncio.sleep(1)           # Yes - sleep is awaitable
await client.chat.completions.create(...)  # Yes - network call
await len([1, 2, 3])             # NO - len() is not awaitable
```

**Rule 3:** The one question to ask: "Does this leave my computer?"
- YES (API call, DB query, file read, network) → use `await`
- NO (math, sorting, string operations) → don't use `await`

## 5. What Does `await` Actually Do?

Let's trace what happens step by step:

In [13]:
async def fetch_user():
    print("  1. About to call API...")
    await asyncio.sleep(1)  # ← HERE: Python says "I'll wait. Anyone else need the CPU?"
    print("  3. API responded! (1 second later)")
    return {"name": "Pallab"}

async def fetch_orders():
    print("  2. About to call DB...")
    await asyncio.sleep(1.5)  # ← Python: "Cool, I'll wait here too."
    print("  4. DB responded! (1.5 seconds later)")
    return [{"id": 1}, {"id": 2}]

async def main():
    print("Starting both at the same time:\n")
    start = time.time()
    
    # gather = "start all, wait for all, return all results"
    user, orders = await asyncio.gather(fetch_user(), fetch_orders())
    
    print(f"\n  Got user: {user}")
    print(f"  Got orders: {orders}")
    print(f"  Total time: {time.time() - start:.1f}s (not 2.5s!)")

await main()

Starting both at the same time:

  1. About to call API...
  2. About to call DB...
  3. API responded! (1 second later)
  4. DB responded! (1.5 seconds later)

  Got user: {'name': 'Pallab'}
  Got orders: [{'id': 1}, {'id': 2}]
  Total time: 1.5s (not 2.5s!)


## 6. `await` vs no `await` — What goes wrong

In [14]:
async def get_data():
    await asyncio.sleep(0.1)
    return 42

# CORRECT: await it
result = await get_data()
print(f"With await:    result = {result}, type = {type(result)}")

# WRONG: forget to await
result = get_data()  # No await!
print(f"Without await: result = {result}, type = {type(result)}")
print("  ^ You got a coroutine OBJECT, not the actual value!")
print("  It's like getting a ticket number instead of your food.")
print("  'await' is what exchanges the ticket for the actual result.")

# Clean up the unawaited coroutine
result.close()

With await:    result = 42, type = <class 'int'>
Without await: result = <coroutine object get_data at 0x1089b2680>, type = <class 'coroutine'>
  ^ You got a coroutine OBJECT, not the actual value!
  It's like getting a ticket number instead of your food.
  'await' is what exchanges the ticket for the actual result.


## 7. Real-World Example: Web Server

In [ ]:
# Simulating a web server handling 5 users

async def handle_request(user_id):
    """Handle one user's request (calls an LLM API)."""
    print(f"  User {user_id}: request received")
    await asyncio.sleep(2)  # Simulate LLM API call (2 seconds)
    print(f"  User {user_id}: response sent")

# --- SYNC server (one at a time) ---
print("SYNC server (handles users one by one):")
start = time.time()
for i in range(5):
    # Can't actually run sync here in jupyter, so simulate:
    pass
print(f"  Would take: 5 users x 2s = 10 seconds total\n")

# --- ASYNC server (all at once) ---
print("ASYNC server (handles all users concurrently):")
start = time.time()

#Notes
# The * unpacks the list into separate arguments. Think of it as "removing the brackets":
# my_list = [handle_request(1), handle_request(2), handle_request(3)]

# # These two are identical:
# await asyncio.gather(*my_list)
# await asyncio.gather(handle_request(1), handle_request(2), handle_request(3))

await asyncio.gather(*[handle_request(i+1) for i in range(5)])
print(f"  Actual time: {time.time() - start:.1f}s (all 5 waited in parallel!)")

SYNC server (handles users one by one):
  Would take: 5 users x 2s = 10 seconds total

ASYNC server (handles all users concurrently):
  User 1: request received
  User 2: request received
  User 3: request received
  User 4: request received
  User 5: request received
  User 1: response sent
  User 2: response sent
  User 3: response sent
  User 4: response sent
  User 5: response sent
  Actual time: 2.0s (all 5 waited in parallel!)


---
# PART B: async for + yield (Async Generators)
---

## 8. The Sync World (normal `for` + `yield`)

In sync code, when you wait for something, the ENTIRE program freezes.
No one else gets served. Like a single cashier — everyone waits in line.

In [16]:
# A sync generator that simulates fetching data from a slow source
def get_items_sync():
    """Pretend each item takes 1 second to arrive from the network."""
    for i in range(4):
        time.sleep(1)  # BLOCKS everything. Nothing else can happen.
        yield f"Item {i + 1}"

# Consuming it
print("Sync: Each item blocks for 1 second...")
start = time.time()

for item in get_items_sync():
    print(f"  [{time.time() - start:.1f}s] Got: {item}")

print(f"\nTotal: {time.time() - start:.1f}s")
print("During those waits, NOTHING else could run.")

Sync: Each item blocks for 1 second...
  [1.0s] Got: Item 1
  [2.0s] Got: Item 2
  [3.0s] Got: Item 3
  [4.0s] Got: Item 4

Total: 4.0s
During those waits, NOTHING else could run.


## 9. The Async World (`async for` + `yield`)

In async code, when you wait for something, Python says "I'm free!" and can do other work.
Like a cashier who helps the next person while your credit card processes.

In [17]:
# An async generator — same thing but with async
async def get_items_async():
    """Pretend each item takes 1 second to arrive from the network."""
    for i in range(4):
        await asyncio.sleep(1)  # Waits, but FREES UP the CPU for others
        yield f"Item {i + 1}"

# Consuming it
print("Async: Each item waits 1 second, but others can run during the wait...")
start = time.time()

async for item in get_items_async():
    print(f"  [{time.time() - start:.1f}s] Got: {item}")

print(f"\nTotal: {time.time() - start:.1f}s")
print("Same time — but during waits, other tasks COULD have run!")

Async: Each item waits 1 second, but others can run during the wait...
  [1.0s] Got: Item 1
  [2.0s] Got: Item 2
  [3.0s] Got: Item 3
  [4.0s] Got: Item 4

Total: 4.0s
Same time — but during waits, other tasks COULD have run!


## 10. Proof: Async Frees Up the CPU

Let's run TWO async generators at the same time. They share the wait time!

In [18]:
async def user_request(user_name, delay):
    """Simulate a user waiting for LLM tokens."""
    print(f"  {user_name}: started")
    for i in range(3):
        await asyncio.sleep(delay)  # Network wait — frees CPU
        print(f"  {user_name}: got token {i + 1}")
    print(f"  {user_name}: done!")


# Run 3 users at the SAME TIME (concurrent)
print("Three users requesting simultaneously:\n")
start = time.time()

await asyncio.gather(
    user_request("Alice", 0.5),
    user_request("Bob",   0.7),
    user_request("Carol", 0.6),
)

print(f"\nAll 3 done in {time.time() - start:.1f}s (NOT 5.4s!)")
print("They shared the waiting time because async frees up during waits.")

Three users requesting simultaneously:

  Alice: started
  Bob: started
  Carol: started
  Alice: got token 1
  Carol: got token 1
  Bob: got token 1
  Alice: got token 2
  Carol: got token 2
  Bob: got token 2
  Alice: got token 3
  Alice: done!
  Carol: got token 3
  Carol: done!
  Bob: got token 3
  Bob: done!

All 3 done in 2.1s (NOT 5.4s!)
They shared the waiting time because async frees up during waits.


## 11. Why `async` Appears in 3 Places

```python
async def stream_tokens():              # ← async #1
    async for chunk in get_chunks():    # ← async #2
        yield chunk

async for token in stream_tokens():    # ← async #3
    print(token)
```

Each one has a different job:

In [19]:
# Let's build it step by step to see each async's role:

# =============================================
# LEVEL 1: The data source (like Groq API)
# =============================================
async def groq_api_stream():
    """
    Simulates Groq sending tokens over the network.
    Each token takes time to generate + travel over network.
    
    'async def' → "This function has waits inside"
    'yield'     → "Give one piece at a time"
    Together    → async generator
    """
    tokens = ["Python", " is", " a", " great", " language"]
    for token in tokens:
        await asyncio.sleep(0.3)  # Network delay
        yield token

print("Level 1: groq_api_stream()")
print("  Like the Groq API — gives tokens one at a time over network")
print("  'async def' because it has 'await' inside")
print("  'yield' because it gives one token at a time")

Level 1: groq_api_stream()
  Like the Groq API — gives tokens one at a time over network
  'async def' because it has 'await' inside
  'yield' because it gives one token at a time


In [20]:
# =============================================
# LEVEL 2: Our processing function
# =============================================
async def stream_tokens():
    """
    Gets tokens from API, processes them, passes along.
    
    async def          → #1: "I have waits inside"
    async for chunk in → #2: "The source delivers items with network waits"
    yield chunk        →     "Pass each piece to whoever called me"
    """
    async for chunk in groq_api_stream():  # async #2: source is async
        yield chunk.upper()  # Transform and pass along

print("Level 2: stream_tokens()")
print("  Consumes from Level 1, transforms, passes up")
print("  'async for' because groq_api_stream() is an async generator")
print("  Can't use normal 'for' — would get TypeError")

Level 2: stream_tokens()
  Consumes from Level 1, transforms, passes up
  'async for' because groq_api_stream() is an async generator
  Can't use normal 'for' — would get TypeError


In [21]:
# =============================================
# LEVEL 3: The consumer (your endpoint / main code)
# =============================================
# async for token in → #3: "stream_tokens() is async, so I must use async for"

print("Level 3: consuming the stream\n")
print("Response: ", end="")

async for token in stream_tokens():  # async #3: source is async
    print(token, end="", flush=True)

print("\n\nEach token appeared with a 0.3s delay (simulated network).")
print("During each delay, other users could have been served!")

Level 3: consuming the stream

Response: PYTHON IS A GREAT LANGUAGE

Each token appeared with a 0.3s delay (simulated network).
During each delay, other users could have been served!


## 12. What Happens if You REMOVE One Async?

In [ ]:
# ERROR 1: Remove 'async' from the function definition
try:
    exec("""
def bad_function():
    await asyncio.sleep(1)
    yield "hello"
""")
except SyntaxError as e:
    print(f"Error without 'async def': {e}")
    print("  → You MUST use 'async def' if the function has 'await' inside.\n")

In [ ]:
# ERROR 2: Use normal 'for' on an async generator
try:
    for token in stream_tokens():  # Wrong! stream_tokens() is async
        print(token)
except TypeError as e:
    print(f"Error using normal 'for' on async generator:")
    print(f"  {e}\n")
    print("  → You MUST use 'async for' to consume an async generator.")
    print("  It's like trying to plug a US charger into a UK outlet.")
    print("  Async producers need async consumers.")

## 13. The Chain Rule

Async is **contagious** — it propagates up the chain:

```
Network wait (Groq API)    → must be async
    ↓ consumed by
stream_tokens()            → must be async (because source is async)
    ↓ consumed by  
Your endpoint              → must use async for (because source is async)
```

If ANYTHING in the chain waits on the network, EVERYTHING above it becomes async.

It's like a relay race — if one runner needs special shoes (async), 
every runner after them also needs special shoes to accept the baton.

---
# PART C: `await` alone vs `async for` + `yield`

**Question:** I see in one place you use just `await` and in another there are all 3 asyncs. When do I use which?

---

## 14. `await` alone — When you get ONE complete result

Use `await` when the function goes away, does its work, and comes back with **the whole answer at once**.

Like ordering food for delivery — you wait, doorbell rings, you get the full meal.

In [ ]:
# === await alone: ONE complete result ===

async def get_full_answer(question):
    """Calls API, waits, gets the COMPLETE answer at once."""
    print(f"  Asking: {question}")
    print(f"  Waiting for full response...")
    await asyncio.sleep(2)  # Simulate API call
    return "Python is a high-level programming language created by Guido van Rossum."

# Usage: just await it
print("Using 'await' — get one complete result:\n")
start = time.time()
answer = await get_full_answer("What is Python?")
print(f"  Got full answer after {time.time() - start:.1f}s:")
print(f"  '{answer}'")
print(f"\n  You waited 2s, then got EVERYTHING at once.")

## 15. `async for` + `yield` — When you get MANY pieces over time

Use `async for` + `yield` when the result comes in **multiple pieces over time** and you want each piece **immediately**.

Like a live cricket score — updates keep coming, one ball at a time.

In [22]:
# === async for + yield: MANY pieces over time ===

async def stream_answer(question):                   # async #1: has waits inside
    """Yields tokens one at a time as they're generated."""
    answer = "Python is a high-level programming language created by Guido van Rossum."
    words = answer.split(" ")
    
    for i, word in enumerate(words):
        await asyncio.sleep(0.2)  # Each token takes time to arrive
        token = f" {word}" if i > 0 else word
        yield token                                  # Give one piece NOW

# Usage: async for to consume it
print("Using 'async for' + 'yield' — get pieces one at a time:\n")
print("  Answer: ", end="")
start = time.time()

async for token in stream_answer("What is Python?"):  # async #3: source is async
    print(token, end="", flush=True)

print(f"\n\n  Total: {time.time() - start:.1f}s")
print(f"  But first word appeared in ~0.2s, not 2.4s!")

Using 'async for' + 'yield' — get pieces one at a time:

  Answer: Python is a high-level programming language created by Guido van Rossum.

  Total: 2.2s
  But first word appeared in ~0.2s, not 2.4s!


## 16. Same API call — two ways

This is what it looks like with a real LLM API (simulated here):

In [ ]:
# Simulating the OpenAI/Groq client

class FakeResponse:
    """Simulates: response.choices[0].message.content"""
    def __init__(self, text):
        self.choices = [type('obj', (object,), {'message': type('obj', (object,), {'content': text})()})()]

class FakeChunk:
    """Simulates: chunk.choices[0].delta.content"""
    def __init__(self, text):
        self.choices = [type('obj', (object,), {'delta': type('obj', (object,), {'content': text})()})()]

class FakeClient:
    """Simulates the OpenAI client."""
    
    async def create_completion(self, messages, stream=False):
        answer = "Python is great for AI and web development."
        
        if not stream:
            # Non-streaming: wait for everything, return complete response
            await asyncio.sleep(2)
            return FakeResponse(answer)
        else:
            # Streaming: return an async generator of chunks
            return self._stream_chunks(answer)
    
    async def _stream_chunks(self, answer):
        words = answer.split(" ")
        for i, word in enumerate(words):
            await asyncio.sleep(0.25)
            token = f" {word}" if i > 0 else word
            yield FakeChunk(token)

client = FakeClient()
print("Fake LLM client created. Now let's use it both ways...\n")

In [ ]:
# ========================================
# WAY 1: await — "Give me the FULL answer"
# ========================================
print("WAY 1: await (stream=False)")
print("  Waiting", end="")
start = time.time()

response = await client.create_completion(
    messages=[{"role": "user", "content": "What is Python?"}],
    stream=False,  # ← No streaming. Wait for everything.
)

# response.choices[0].message.content = full answer
full_answer = response.choices[0].message.content

print(f" ({time.time() - start:.1f}s)")
print(f"  Got: '{full_answer}'")
print(f"  Waited 2s seeing NOTHING, then got everything.\n")

In [ ]:
# ========================================
# WAY 2: async for — "Give me each PIECE"
# ========================================
print("WAY 2: async for (stream=True)")
print("  Answer: ", end="")
start = time.time()

stream = await client.create_completion(
    messages=[{"role": "user", "content": "What is Python?"}],
    stream=True,  # ← Streaming! Get tokens as they come.
)

# chunk.choices[0].delta.content = one token at a time
async for chunk in stream:
    token = chunk.choices[0].delta.content
    print(token, end="", flush=True)

print(f"\n  Total: {time.time() - start:.1f}s")
print(f"  First word appeared in ~0.25s, not 2s!")

## 17. Decision Table: When to Use Which

| Situation | Use | Analogy |
|---|---|---|
| One result, wait for it | `await` | Order food delivery — wait, get full meal |
| Many results over time | `async for` + `yield` | Live score — updates arrive one by one |

**Examples:**

| Task | Use `await` | Use `async for` |
|---|---|---|
| Get user from DB | `user = await db.get(id)` | - |
| Stream LLM tokens | - | `async for chunk in stream` |
| Fetch a web page | `html = await fetch(url)` | - |
| Read large file chunk by chunk | - | `async for line in file` |
| Send an email | `await send_email()` | - |
| WebSocket messages | - | `async for msg in websocket` |

**Short version:**
- `await` = "give me the whole pizza when it's done"
- `async for` + `yield` = "give me each slice as it comes out of the oven"

---
# PART D: Summary
---

## Quick Reference

### async/await:
| Keyword | Meaning | When to use |
|---|---|---|
| `async def` | "This function has waits" | Any function that calls `await` or `async for` |
| `await` | "Do this, but free CPU while waiting" | Network calls, DB queries, file I/O |
| `asyncio.gather()` | "Run all these concurrently" | Multiple independent async tasks |

### async for + yield:
| Keyword | Meaning | When to use |
|---|---|---|
| `async def` + `yield` | Async generator | Producing items that involve network waits |
| `async for x in gen()` | Async consumer | Consuming an async generator |
| `yield` | "Give one piece now" | Same as sync — yield itself is never async |

### The one rule:
**"Does this leave my computer?"**
- YES → `await` it
- NO → don't

### Mental model:
```
await = "Go do this task. While you're gone, I'll help others. Tell me when you're back."
```

That's it. That's all async is.